# M05 — Understand Arrays by Making Python Too Slow

**Objective:** experience why arrays and vectorization matter computationally.

Whole-first loop:

**useful Python-loop computation → scale safely → predict timing → time loop → vectorize with NumPy → compare correctness → compare timing → inspect shapes/broadcasting → generalize**

This notebook is deterministic, CPU-only, network-free, and uses no paid API or secret. Timing values will vary by machine.

## 1. Start with a useful computation

A small retailer needs a total for every order. Each order has quantities for four products. The rule is:

1. multiply each quantity by its product price;
2. add line values to get the gross subtotal;
3. apply the order's discount;
4. add shipping when the discounted subtotal is below 45.

**Predict before running:** calculate all three final totals by hand. Mark which order pays shipping.

In [ ]:
from time import perf_counter

import numpy as np

FREE_SHIPPING_THRESHOLD = 45.0
SHIPPING_FEE = 4.5

def python_order_totals(units_rows, price_values, discount_values):
    """Compute order totals with Python-level loops."""
    totals = []
    for units, discount_rate in zip(units_rows, discount_values):
        gross_subtotal = 0.0
        for quantity, price in zip(units, price_values):
            gross_subtotal += float(quantity) * float(price)
        discounted_subtotal = gross_subtotal * (1.0 - float(discount_rate))
        shipping = SHIPPING_FEE if discounted_subtotal < FREE_SHIPPING_THRESHOLD else 0.0
        totals.append(discounted_subtotal + shipping)
    return totals

def vectorized_order_components(units_rows, price_values, discount_values):
    """Return inspectable NumPy intermediates for the same pricing rule."""
    units_array = np.asarray(units_rows)
    prices_array = np.asarray(price_values, dtype=np.float64)
    discounts_array = np.asarray(discount_values, dtype=np.float64)

    if units_array.ndim != 2:
        raise ValueError("units must have shape (orders, products)")
    if prices_array.shape != (units_array.shape[1],):
        raise ValueError("prices must have shape (products,)")
    if discounts_array.shape != (units_array.shape[0],):
        raise ValueError("discounts must have shape (orders,)")

    line_values = units_array * prices_array
    discount_columns = discounts_array[:, np.newaxis]
    discounted_lines = line_values * (1.0 - discount_columns)
    discounted_subtotals = discounted_lines.sum(axis=1)
    shipping = np.where(
        discounted_subtotals < FREE_SHIPPING_THRESHOLD,
        SHIPPING_FEE,
        0.0,
    )
    totals = discounted_subtotals + shipping
    return {
        "units": units_array,
        "prices": prices_array,
        "discounts": discounts_array,
        "line_values": line_values,
        "discount_columns": discount_columns,
        "discounted_lines": discounted_lines,
        "discounted_subtotals": discounted_subtotals,
        "shipping": shipping,
        "totals": totals,
    }

def vectorized_order_totals(units_rows, price_values, discount_values):
    return vectorized_order_components(
        units_rows, price_values, discount_values
    )["totals"]

def best_seconds(function, *args, repeats=3):
    """Return minimum duration, all durations, and the final result."""
    durations = []
    result = None
    for _ in range(repeats):
        start = perf_counter()
        result = function(*args)
        durations.append(perf_counter() - start)
    return min(durations), durations, result

In [ ]:
product_names = ["tea", "coffee", "oats", "oil"]
small_prices_list = [12.5, 8.0, 5.5, 20.0]
small_units_list = [
    [2, 1, 0, 1],
    [0, 3, 2, 0],
    [5, 0, 4, 1],
]
small_discounts_list = [0.10, 0.00, 0.05]

loop_small_totals = python_order_totals(
    small_units_list, small_prices_list, small_discounts_list
)
expected_small_totals = [47.7, 39.5, 99.275]

print("Python-loop totals:", loop_small_totals)
np.testing.assert_allclose(loop_small_totals, expected_small_totals, rtol=0, atol=1e-12)

The loop is useful and readable. It also asks Python to dispatch multiplication, addition, conversion, iteration, and list appends element by element. That overhead becomes visible when the number of orders grows.

**Explain before scaling:** if there are `n` orders and `p` products, how many quantity-price multiplications does the nested loop perform?

## 2. Scale the same workload safely

We will generate exactly 200,000 orders with a fixed random seed. The product count stays at four, and no data leave this process.

**Timing prediction — write this down before running the next cells:**

- predicted Python-loop seconds: ______
- predicted NumPy seconds: ______
- predicted faster representation and reason: ______
- result tolerance you expect to need: ______

The prediction is part of the experiment; accuracy is not the grading target.

In [ ]:
SCALE_ROWS = 200_000
rng = np.random.default_rng(20260815)
scaled_units = rng.integers(0, 8, size=(SCALE_ROWS, 4), dtype=np.int64)
scaled_prices = np.asarray(small_prices_list, dtype=np.float64)
scaled_discounts = rng.choice(
    np.array([0.00, 0.05, 0.10, 0.15], dtype=np.float64),
    size=SCALE_ROWS,
)

# Build both representations before either timer starts.
scaled_units_list = scaled_units.tolist()
scaled_prices_list = scaled_prices.tolist()
scaled_discounts_list = scaled_discounts.tolist()

assert scaled_units.shape == (SCALE_ROWS, 4)
assert scaled_discounts.shape == (SCALE_ROWS,)
print("bounded workload:", scaled_units.shape, scaled_units.dtype)

### Time the loop first

Input generation and representation conversion are outside the timed function. We repeat the calculation and retain the minimum duration to reduce interference from unrelated pauses.

In [ ]:
loop_seconds, loop_durations, loop_scaled_totals = best_seconds(
    python_order_totals,
    scaled_units_list,
    scaled_prices_list,
    scaled_discounts_list,
    repeats=3,
)
print("loop durations (seconds):", [round(value, 6) for value in loop_durations])
print("best loop seconds:", round(loop_seconds, 6))
assert len(loop_scaled_totals) == SCALE_ROWS

## 3. Vectorize without changing the rule

Represent quantities as `(orders, products)`, prices as `(products,)`, and discounts as `(orders,)`. NumPy can apply the product-price arithmetic across the entire homogeneous buffer.

**Predict before running:**

- What is the shape of `units * prices`?
- Why does a price vector of shape `(4,)` align with the product axis?
- What shape must per-order discounts have to multiply every product in its own order?

In [ ]:
small_units = np.asarray(small_units_list, dtype=np.int64)
small_prices = np.asarray(small_prices_list, dtype=np.float64)
small_discounts = np.asarray(small_discounts_list, dtype=np.float64)

small_components = vectorized_order_components(
    small_units, small_prices, small_discounts
)
numpy_small_totals = small_components["totals"]

print("NumPy totals:", numpy_small_totals)
np.testing.assert_allclose(numpy_small_totals, loop_small_totals, rtol=0, atol=1e-12)

### Correctness before speed

**Predict before running:** Should two implementations of floating-point arithmetic be required to match bit for bit, or within a declared tolerance? Write a reason.

In [ ]:
numpy_scaled_totals = vectorized_order_totals(
    scaled_units, scaled_prices, scaled_discounts
)
loop_scaled_array = np.asarray(loop_scaled_totals, dtype=np.float64)
maximum_absolute_difference = np.max(
    np.abs(loop_scaled_array - numpy_scaled_totals)
)
np.testing.assert_allclose(
    numpy_scaled_totals, loop_scaled_array, rtol=1e-12, atol=1e-12
)
print("result shape:", numpy_scaled_totals.shape)
print("maximum absolute difference:", maximum_absolute_difference)

In [ ]:
numpy_seconds, numpy_durations, numpy_timed_totals = best_seconds(
    vectorized_order_totals,
    scaled_units,
    scaled_prices,
    scaled_discounts,
    repeats=3,
)
np.testing.assert_allclose(
    numpy_timed_totals, loop_scaled_array, rtol=1e-12, atol=1e-12
)
print("NumPy durations (seconds):", [round(value, 6) for value in numpy_durations])
print("best NumPy seconds:", round(numpy_seconds, 6))

In [ ]:
observed_speedup = loop_seconds / numpy_seconds
timing_report = {
    "rows": SCALE_ROWS,
    "products": scaled_units.shape[1],
    "loop_seconds": loop_seconds,
    "numpy_seconds": numpy_seconds,
    "observed_speedup": observed_speedup,
}
print({key: round(value, 6) if isinstance(value, float) else value
       for key, value in timing_report.items()})
assert loop_seconds > 0 and numpy_seconds > 0

# No fixed speedup assertion: timings are observations tied to this run and machine.

Compare the recorded prediction with `timing_report`. Explain the gap using evidence from this run. Do not generalize one ratio into a universal claim: workload shape, dtype, hardware, NumPy build, memory pressure, and the exact operation all matter.

## 4. Inspect what made the array computation possible

Now attach vocabulary to the working whole.

- **shape**: length along each axis;
- **dtype**: common element representation;
- **axis**: a dimension of the array; an aggregation removes or retains axes as requested;
- **broadcasting**: combining compatible shapes by expanding dimensions of size one without materializing repeated input data.

In [ ]:
for name in [
    "units", "prices", "discounts", "line_values",
    "discount_columns", "discounted_lines", "discounted_subtotals",
    "shipping", "totals",
]:
    value = small_components[name]
    print(f"{name:22s} shape={value.shape!s:8s} dtype={value.dtype}")

assert small_components["units"].dtype == np.dtype(np.int64)
assert small_components["line_values"].dtype == np.dtype(np.float64)
assert small_components["discount_columns"].shape == (3, 1)

### Indexing and slicing

**Predict before running:** What values and shapes result from `small_units[0, 1]`, `small_units[:2, 1:3]`, and selecting totals greater than or equal to 45?

In [ ]:
one_quantity = small_units[0, 1]
rectangular_slice = small_units[:2, 1:3]
high_value_mask = numpy_small_totals >= 45.0
high_value_totals = numpy_small_totals[high_value_mask]

print("indexed scalar:", one_quantity)
print("slice and shape:", rectangular_slice.tolist(), rectangular_slice.shape)
print("boolean mask:", high_value_mask)
print("selected totals:", high_value_totals)

assert one_quantity == 1
np.testing.assert_array_equal(rectangular_slice, [[1, 0], [3, 2]])
np.testing.assert_array_equal(high_value_mask, [True, False, True])

### Aggregation and axis meaning

`line_values` has shape `(orders, products)`.

**Predict before running:**

- Which axis must disappear to produce one subtotal per order?
- Which axis must disappear to produce one revenue total per product?
- What are the two result shapes?

In [ ]:
line_values = small_components["line_values"]
per_order_gross = line_values.sum(axis=1)
per_product_revenue = line_values.sum(axis=0)

print("per-order gross, axis=1:", per_order_gross, per_order_gross.shape)
print("per-product revenue, axis=0:", per_product_revenue, per_product_revenue.shape)

np.testing.assert_allclose(per_order_gross, [53.0, 35.0, 104.5])
np.testing.assert_allclose(per_product_revenue, [87.5, 32.0, 33.0, 40.0])
assert per_order_gross.shape == (3,)
assert per_product_revenue.shape == (4,)

`units * prices`, `line_values * (1 - discount_columns)`, the comparison with the shipping threshold, `np.where`, and addition of shipping are all vectorized arithmetic. Python invokes each array operation once; compiled NumPy loops over homogeneous buffers internally.

## 5. Controlled failure A — incompatible broadcasting

The line values have shape `(3, 4)`. The discount rates have shape `(3,)`.

**Predict before running:** align the shapes from the right. Which two trailing dimensions conflict in `line_values * (1 - small_discounts)`? Write the expected exception type.

In [ ]:
try:
    line_values * (1.0 - small_discounts)
except ValueError as error:
    broadcast_failure = {
        "type": type(error).__name__,
        "line_values_shape": line_values.shape,
        "discount_shape": small_discounts.shape,
        "message": str(error),
    }
else:
    raise AssertionError("Expected incompatible broadcasting to raise ValueError")

print(broadcast_failure)
assert broadcast_failure["type"] == "ValueError"

In [ ]:
discount_columns = small_discounts[:, np.newaxis]
repaired_discounted_lines = line_values * (1.0 - discount_columns)

print("repair shapes:", line_values.shape, discount_columns.shape, repaired_discounted_lines.shape)
assert discount_columns.shape == (3, 1)
assert repaired_discounted_lines.shape == (3, 4)
np.testing.assert_allclose(
    repaired_discounted_lines, small_components["discounted_lines"]
)

## 6. Controlled failure B — valid syntax, wrong axis

A wrong axis can return valid numbers with the wrong meaning.

**Predict before running:** What shape does `line_values.sum(axis=0)` return, and why can it not be one total per order here?

In [ ]:
wrong_axis_totals = line_values.sum(axis=0)
try:
    assert wrong_axis_totals.shape == (small_units.shape[0],), (
        "expected one value per order",
        wrong_axis_totals.shape,
    )
except AssertionError as error:
    axis_failure = {
        "type": type(error).__name__,
        "observed_shape": wrong_axis_totals.shape,
        "expected_shape": (small_units.shape[0],),
    }
else:
    raise AssertionError("Expected the semantic axis shape check to fail")

print(axis_failure)
assert axis_failure["observed_shape"] == (4,)

In [ ]:
repaired_order_gross = line_values.sum(axis=1)
assert repaired_order_gross.shape == (small_units.shape[0],)
np.testing.assert_allclose(repaired_order_gross, [53.0, 35.0, 104.5])
print("repaired per-order shape:", repaired_order_gross.shape)

## 7. No-AI Gate — manual prediction followed by implementation

Complete the prediction on paper without AI-generated code. Do not run the implementation cell until the prediction exists.

For quantities `[[2, 1, 0], [0, 3, 2]]` and prices `[10.0, 4.0, 1.5]`, manually write:

1. the shape and every value of the elementwise product;
2. the two per-order subtotals;
3. the three per-product totals;
4. the boolean mask for order subtotals greater than or equal to 20.

Prediction recorded at: ______

Only then run the next cell.

In [ ]:
gate_units = np.array([[2, 1, 0], [0, 3, 2]], dtype=np.int64)
gate_prices = np.array([10.0, 4.0, 1.5], dtype=np.float64)
gate_line_values = gate_units * gate_prices
gate_order_subtotals = gate_line_values.sum(axis=1)
gate_product_totals = gate_line_values.sum(axis=0)
gate_mask = gate_order_subtotals >= 20.0

np.testing.assert_allclose(gate_line_values, [[20.0, 4.0, 0.0], [0.0, 12.0, 3.0]])
np.testing.assert_allclose(gate_order_subtotals, [24.0, 15.0])
np.testing.assert_allclose(gate_product_totals, [20.0, 16.0, 3.0])
np.testing.assert_array_equal(gate_mask, [True, False])
print("manual prediction verified against implementation")

## 8. Generalize to a new shape

A vectorized rule is not general merely because it works on the teaching fixture. This transfer uses four orders and three products.

**Predict before running:** write every intermediate shape and identify which code should remain unchanged.

In [ ]:
transfer_units = np.array(
    [[1, 0, 2], [3, 1, 0], [0, 4, 1], [2, 2, 2]],
    dtype=np.int64,
)
transfer_prices = np.array([7.0, 3.5, 11.0], dtype=np.float64)
transfer_discounts = np.array([0.00, 0.05, 0.10, 0.15], dtype=np.float64)

transfer_loop = python_order_totals(
    transfer_units.tolist(),
    transfer_prices.tolist(),
    transfer_discounts.tolist(),
)
transfer_components = vectorized_order_components(
    transfer_units, transfer_prices, transfer_discounts
)
transfer_numpy = transfer_components["totals"]

assert transfer_components["line_values"].shape == (4, 3)
assert transfer_components["discount_columns"].shape == (4, 1)
assert transfer_numpy.shape == (4,)
np.testing.assert_allclose(transfer_numpy, transfer_loop, rtol=1e-12, atol=1e-12)
print("transfer totals:", transfer_numpy)

## 9. Generalize the lesson

Write a short explanation in your own words:

1. Why was the Python loop a good first implementation?
2. Which work moved out of Python during vectorization?
3. How did shape and axis meaning make the compact expression trustworthy?
4. Which correctness check had to pass before interpreting speed?
5. When might a loop still be clearer or more appropriate?
6. What new shape would you test next to challenge your assumptions?

In [ ]:
# Executable end-of-mission invariants. These validate code, not learner evidence.
assert SCALE_ROWS == 200_000
assert numpy_scaled_totals.shape == (SCALE_ROWS,)
assert small_components["line_values"].shape == (3, 4)
assert small_components["discount_columns"].shape == (3, 1)
assert broadcast_failure["type"] == "ValueError"
assert axis_failure["type"] == "AssertionError"
np.testing.assert_allclose(numpy_small_totals, expected_small_totals, rtol=0, atol=1e-12)
print("M05 computational invariants passed")